In [1]:
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings("ignore")

# ==================== LOAD DATA ====================
print("🔄 Loading data...")
df = pd.read_csv('data/processed/filtered_complaints.csv', low_memory=False)
df = df.reset_index(drop=True)   # <-- Important fix

print(f"Loaded {len(df):,} complaints")
print("Product column exists:", 'Product' in df.columns)

# Sample without groupby lambda (safer)
print("\nPerforming sampling...")
sample_size = 12000

# Get value counts
product_counts = df['Product'].value_counts()

# Create balanced sample
df_sample = pd.DataFrame()
for product, count in product_counts.items():
    n_samples = max(50, int(sample_size * count / len(df)))
    n_samples = min(n_samples, count)
    subset = df[df['Product'] == product].sample(n=n_samples, random_state=42)
    df_sample = pd.concat([df_sample, subset])

df_sample = df_sample.reset_index(drop=True)

print(f"✅ Sample size: {len(df_sample):,}")
print("Product distribution in sample:")
print(df_sample['Product'].value_counts())

# ==================== CHUNKING ====================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
)

chunks = []
metadatas = []

print("\nCreating chunks...")
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    text = str(row.get('cleaned_narrative', ''))
    if len(text.strip()) < 30:
        continue
    splits = text_splitter.split_text(text)
    for i, chunk in enumerate(splits):
        chunks.append(chunk)
        metadatas.append({
            "complaint_id": str(row.get('Complaint ID', idx)),
            "product_category": str(row.get('Product', '')),
            "issue": str(row.get('Issue', '')),
            "sub_issue": str(row.get('Sub-issue', '')),
            "company": str(row.get('Company', '')),
            "state": str(row.get('State', '')),
            "date_received": str(row.get('Date received', '')),
            "chunk_index": i,
            "total_chunks": len(splits)
        })

print(f"Total chunks: {len(chunks):,}")

# ==================== EMBEDDINGS ====================
print("\nGenerating embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, batch_size=64, show_progress_bar=True)

os.makedirs('vector_store', exist_ok=True)
np.save('vector_store/sample_embeddings.npy', embeddings)
print("✅ Embeddings saved.")

# ==================== CHROMADB ====================
print("\nCreating ChromaDB...")
client = chromadb.PersistentClient(path="vector_store/chroma_db")
collection = client.get_or_create_collection(name="complaints")

batch_size = 1000
for i in tqdm(range(0, len(chunks), batch_size)):
    batch_chunks = chunks[i:i+batch_size]
    batch_emb = embeddings[i:i+batch_size].tolist()
    batch_meta = metadatas[i:i+batch_size]
    batch_ids = [f"chunk_{i+j}" for j in range(len(batch_chunks))]
    
    collection.add(
        documents=batch_chunks,
        embeddings=batch_emb,
        metadatas=batch_meta,
        ids=batch_ids
    )

print("🎉✅ Task 2 Completed Successfully!")

c:\Users\user 1\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Loading data...
Loaded 454,472 complaints
Product column exists: True

Performing sampling...
✅ Sample size: 12,008
Product distribution in sample:
Product
Checking or savings account                                3705
Credit card or prepaid card                                2869
Money transfer, virtual currency, or money service         2566
Credit card                                                2129
Payday loan, title loan, or personal loan                   455
Payday loan, title loan, personal loan, or advance loan     234
Money transfers                                              50
Name: count, dtype: int64

Creating chunks...


100%|██████████| 12008/12008 [00:07<00:00, 1622.31it/s]


Total chunks: 36,556

Generating embeddings...


Batches: 100%|██████████| 572/572 [35:54<00:00,  3.77s/it] 


✅ Embeddings saved.

Creating ChromaDB...


100%|██████████| 37/37 [02:14<00:00,  3.63s/it]

🎉✅ Task 2 Completed Successfully!
